# Day 46 — Deep Learning overview (PyTorch/TensorFlow)
Objectives:
- Understand tensors, computation graphs, autograd.
- Choose a framework; we'll demo PyTorch.
- Hello-world linear model training loop.
Note: Install PyTorch per your platform instructions.

In [ ]:
import torch
torch.__version__
# Simple linear regression y = wx + b
X = torch.linspace(-1,1,200).unsqueeze(1)
true_w, true_b = 2.0, -0.5
y = true_w*X + true_b + 0.1*torch.randn_like(X)
model = torch.nn.Sequential(torch.nn.Linear(1,1))
opt = torch.optim.SGD(model.parameters(), lr=0.1)
loss_fn = torch.nn.MSELoss()
for epoch in range(200):
    opt.zero_grad(); pred = model(X); loss = loss_fn(pred,y); loss.backward(); opt.step()
loss.item(), list(model.parameters())


## How to use this notebook

Select the `Python (ds60sqlpy)` kernel, start at the first cell, and
write each prediction before execution. Keep attempts in the
provided scratch cell or new cells. Restart the kernel and run from
the top before calling the work reproducible.

## Concept lab — tensors, computation graphs, gradients, and the optimizer cycle

### Mental model

A neural network is a parameterized function composed from tensor
operations. The **forward pass** produces predictions and a loss.
Autograd records the computation graph and `backward()` applies the
chain rule to accumulate gradients in parameter `.grad` fields.
`optimizer.step()` then updates parameters according to those gradients.

Gradients accumulate by default, so a training step normally follows
`zero_grad → forward → loss → backward → step`. Evaluation additionally
switches training-specific module behavior off and disables gradient
tracking. A falling training loss is evidence of optimization, not of
generalization.

### Read the API before running it

- **`tensor.requires_grad_(True)`:** marks a leaf tensor whose derivative should be accumulated.
- **`loss.backward()`:** computes derivatives through the recorded graph and adds them to existing `.grad` values.
- **`optimizer.zero_grad(); ...; optimizer.step()`:** clears old gradients, computes a new step, and updates registered parameters.

For every call, identify input data, learned state, returned value,
and a check that can fail. That habit prevents a successful cell
from being mistaken for a correct analysis.

### Focused example A — differentiate a scalar function by autograd

**Predict first:** write down the expected shape, type, ordering, or
direction of the result. Then run the next cell.

**Assumption:** `x` is a scalar leaf and only one backward pass has contributed to its gradient.

In [ ]:
import torch

x = torch.tensor(3.0, requires_grad=True)
y = x**2 + 2 * x
y.backward()
print({"y": y.item(), "dy_dx": x.grad.item()})
assert x.grad.item() == 8.0  # derivative: 2*x + 2 at x=3

**Expected observation:** Autograd returns derivative 8.0, matching the hand-derived chain rule.

Do not force exact equality for estimates based on samples. Record
the seed, sample size, tolerance, and metric where they matter.

### Focused example B — see gradient accumulation explicitly

This example changes one important condition. Predict how and why
the result should differ from Example A.

**Assumption:** Gradient accumulation is not intentionally being used to combine microbatches.

In [ ]:
import torch

weight = torch.tensor(2.0, requires_grad=True)
(weight**2).backward()
first = weight.grad.item()
(weight**2).backward()
accumulated = weight.grad.item()
weight.grad.zero_()
cleared = weight.grad.item()
print({"first": first, "accumulated": accumulated, "cleared": cleared})
assert (first, accumulated, cleared) == (4.0, 8.0, 0.0)

**Expected observation:** A second backward call adds another gradient; clearing is an explicit part of each ordinary optimization step.

### Debugging and practice ramp

**Common mistake:** Forgetting `zero_grad`, calling `step` before `backward`, or evaluating with dropout active and gradient tracking enabled.

**Diagnostic:** Print tensor shape/dtype/device, `requires_grad`, loss value, gradient norms, parameter deltas, and train/validation curves for one tiny batch.

| Stage | Action | Evidence |
|---|---|---|
| Recall | Define tensors, computation graphs, gradients, and the optimizer cycle in your own words and identify its input and output. | A definition that does not rely on the library name. |
| Predict | Predict the examples before execution, including shape and direction. | A written prediction and an explanation of any mismatch. |
| Implement | Recreate one example with a changed but valid input. | Code plus an assertion for the central invariant. |
| Debug | Trigger the named mistake or edge case intentionally. | The observed symptom and the smallest diagnostic that isolates it. |
| Transfer | Apply the idea to a different local dataset or decision. | A stated assumption, metric, and reason the method is suitable. |

**Stop condition:** Stop training when loss is non-finite, gradients explode/vanish, validation degrades, or the data/label shape contract is uncertain.

Continue to the numbered practice only after you can explain both
examples without rereading their code.

## Learner exercises and progressive hints

1. Plot the loss over epochs.

**Verify:** For task `Plot the loss over epochs`, show the labeled figure and reconcile it with a numeric summary so appearance is not the only check.






2. Replace SGD with Adam and compare convergence.

**Verify:** For task `Replace SGD with Adam and compare convergence`, use identical data, split, metric, and budget for both sides; record a side-by-side result and isolate the condition that changed.






3. Add one hidden layer and a ReLU activation.

**Verify:** For task `Add one hidden layer and a ReLU activation`, demonstrate the concrete requirement “3. Add one hidden layer and a ReLU activation” with explicit inputs, observable output, and one counterexample.







### Progressive hints

1. Append `loss.item()` once per epoch; use a logarithmic y-axis if early values
   obscure later improvement.
2. Recreate the model from the same seed for a fair comparison. Adam and SGD
   generally need different learning rates, so report both settings.
3. A one-input regression MLP can use
   `Linear(1, width) → ReLU() → Linear(width, 1)`.

The separate solution extends these ideas with train/validation curves,
capacity comparisons, and gradient-stability techniques. Treat those as deeper
reference material after completing the notebook's three exercises.

### Additional mastery practice

Trace tensor shapes, gradients, modes, and loss aggregation through a complete training step. Deep-learning code is correct only when its state transitions are explicit.

Predict or plan before you run code. Use the hint only after an honest
attempt, and record the evidence that would prove your result correct.

4. **Autograd tracing:** For one scalar regression batch, annotate every line from `zero_grad()` through `step()`: which tensors receive gradients, when are they accumulated, and when do parameters change?
   **Progressive hint:** Gradients accumulate in parameter `.grad` fields during backward; the optimizer reads them during step. zero_grad clears the previous batch.

**Verify:** For task `Autograd tracing: For one scalar regression batch, annotate every line from zerograd() throug...`, demonstrate the concrete requirement “4. Autograd tracing: For one scalar regression batch, annotate every line from zero grad through step : which tensors receive gradients, when are they accumulated, and when do para” with explicit inputs, observable output, and one counterexample.







5. **Mode debugging:** Build a model with Dropout and BatchNorm, then compare repeated predictions in `train()` and `eval()` modes. Explain why `torch.no_grad()` is related but not interchangeable.
   **Progressive hint:** Mode changes module behavior; no_grad disables graph recording. Validation usually needs both `model.eval()` and `with torch.no_grad()`.

**Verify:** For task `Mode debugging: Build a model with Dropout and BatchNorm, then compare repeated predictions i...`, use identical data, split, metric, and budget for both sides; record a side-by-side result and isolate the condition that changed; then reproduce the failure first, capture its smallest observable symptom, apply one scoped fix, and rerun the failing plus normal case.







6. **Loss-aggregation edge case:** Compare averaging per-batch losses with a sample-weighted epoch loss when the final batch is smaller. Implement the correct aggregation.
   **Progressive hint:** Multiply each mean batch loss by batch size, sum, then divide by the number of examples.

**Verify:** For task `Loss-aggregation edge case: Compare averaging per-batch losses with a sample-weighted epoch l...`, assert the return type/shape/value for the stated valid input and assert the named boundary or invalid input raises/returns exactly the documented behavior; then use identical data, split, metric, and budget for both sides; record a side-by-side result and isolate the condition that changed.






Before opening the reference solution, explain the relevant assumption,
failure mode, and validation check for every answer.

In [ ]:
# Expanded mastery lab scratch space
#
# Keep the official solution closed until you have attempted each task.
# Add small assertions, shape checks, or metric comparisons as evidence.

# Practice 4 — Autograd tracing


# Practice 5 — Mode debugging


# Practice 6 — Loss-aggregation edge case
